## Evaluation 1: internal evlauation on the generated word assoications: 

Below is the instruction:

1.1. Get a list of cue words that are in the test set (perhaps 1.1 and 1.2 for Sukai Huang? what's your availaablity recently? I can give you the list that I found interesting, you do the interesect with your test set; or just get the associations in the test set, we see which ones are interesting). I shared the samples that I was playing around in the excel: 250605_playground_trained_word_association_models.xlsx to give you some ideas. 

1.2 Generate the associations for the list of cues from step 1 (decide which prompt to use, how many associations to generate)

In [1]:
import requests
import json
import os
from openai import OpenAI


In [42]:
def get_reasoning_and_content(messages, 
                             model_type,
                             want_print=False):
    """
    Sends chat messages to a vLLM server and returns a tuple of (reasoning_content, real_content).
    """
    reasoning_flag = False
    if model_type == "ft":
        model="sukai/self_model"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8001/v1"
    elif model_type == "raw":
        model="Qwen/Qwen2.5-7B-Instruct"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8002/v1"
    else:
        reasoning_flag = True
        model="Qwen/Qwen3-32B"
        openai_api_key="EMPTY"
        openai_api_base="http://localhost:8000/v1"
    
    client = OpenAI(
        api_key=openai_api_key,
        base_url=openai_api_base,
    )
    
    params = {
        "model": model,
        "messages": messages,
        "stream": True,
    }
    if not reasoning_flag:
        params["max_tokens"] = 120
    
      
    stream = client.chat.completions.create(
        **params
    )

    reasoning_content = ""
    real_content = ""
    start_reasoning_flg = False
    reasoning_inidicator = False
    count = 0

    for chunk in stream:
        count += 1
        # Handle reasoning content if present
        if chunk.choices and chunk.choices[0].delta and hasattr(chunk.choices[0].delta, "reasoning_content"):
            if not start_reasoning_flg and not reasoning_inidicator:
                if want_print:
                    print("\n=== Reasoning: ===")
                reasoning_inidicator = True
            reasoning_content += chunk.choices[0].delta.reasoning_content
            if want_print:
                print(chunk.choices[0].delta.reasoning_content, end="", flush=True)
            start_reasoning_flg = True
        else:
            if count > 3 and start_reasoning_flg:
                if want_print:
                    print("\n=== End Reasoning ===")
                start_reasoning_flg = False
        # Handle main content
        if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
            real_content += chunk.choices[0].delta.content
            if want_print:
                print(chunk.choices[0].delta.content, end="", flush=True)
    if want_print:
        print()
    return reasoning_content, real_content

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "请帮我根据提示词，将相对应的关联词，关联词已经根据关联频率从高到低排好序了，请你将关联词分成四份，并进行造句"},
]

messages = [
    # {"role": "system", "content": "您是一款专为全面探索词语关联而设计的高级语言模型。"},
    {"role": "system", "content": "You are a helpful assistant."},
    
    {"role": "user", "content": "给定一个提示词，你的任务是生成一个与该提示词相关联的全面词汇列表。目标是尽可能涵盖所有相关的语境、用法和含义，避免重复相似的概念。不要生成受其他词语存在影响的词语，而是专注于提示词本身。提示词：方便。"},
]

print(" ===== test FT llm ====== ")

reasoning, content = get_reasoning_and_content(messages, 'ft')

print(" ===== test raw ====== ")
reasoning, content = get_reasoning_and_content(messages, 'raw')

print(" ===== test powerful =====")
reasoning, content = get_reasoning_and_content(messages, 'power')

 ===== test FT llm ====== 
快捷， 位置， 宝树， 上厕所， 快速， 意大利面， 快， 自行车， 方法， 利用， 树， 考虑， 美食， 生产， 卫生间， 小树， 早餐， 厕所， 简便， 面试， 准则， 菜贩， 移民， 工具， 门， 小卖店， 快递， 安便， 本人， 好的， 面条， 快捷， 男女， 肯德基， 放屁， 通信， 方面， 地理位置， 面粉， 工厂， 万能， 一日七面， 开心， 树木， 轻松， 网速， 不明， 公共卫生间， 方幸， 直观， 普通， 面包， 秋atable， 和， 便利店， 化， 法语， 咖喱， 水疗， 出行， 旅店， 生活， 巧克力， 食物， 辛苦， 面包房， 附麟， 方便面侠， 方方， 路灯， 总结， 借你个厕所， 人的， 健康， 方便面安方便， 方面粉， 日本， 路边， 丸子， 功能， 日语， 面， 结束， 优越， 心境， 手段， 过渡， 意面， 医院， 贺词， 轻便， 十便， 店铺， 六便士， 付费， 商店
 ===== test raw ====== 
1. 方便（形容词） - 描述事物易于获取、操作、理解等特点。

2. 方便面 - 一种加工食品，通常在热水中快速煮熟，便于食用。

3. 方便法 - 简便迅速的方法。

4. 方便工具 - 使操作、搬运、安装等变得更简单的工具。

5. 方便服务 - 提供快捷服务的场所，如便利店、药店等。

6. 货币兑换方便 - 货币兑换快捷、简单，通常在机场、银行、货币兑换点等地点进行。

7. 便利交通 - 交通方便，比如公交、地铁、出租车等交通工具。

8. 电子邮件方便 - 通过电子邮件发送或接收信息便捷迅速。

9. 便携式 - 可携带的，适合快速移动或使用即拿即用。

10. 银行取款方便 - 可在银行柜台、ATM等地点轻松取款。

11. 购物方便 - 购物便捷，可以快速找到所需商品。

12. 电子支付方便 - 通过手机、电脑等设备进行支付便捷快速。

13. 易于操作 - 使用或操作某个事物容易，减轻用户的操作负担。

14. 便利贴 - 一种便利贴纸，记事、便签等用途广泛。

15. 便携洗衣液 - 便于携带和使用的洗衣液，特别是旅行途中或外出时的优选。

16. 便于搬运 - 便于移动或运输，更利于其他用途或需求实现。

17. 便当 - 快速准备、携

In [ ]:
zh_swow_test_data_fp = '/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/llm_swow_finetune_dataset/swow_zh/test/chunk_0.jsonl'

import jsonlines
from tqdm.auto import tqdm
from IPython.display import HTML

In [6]:
data_dict = {} # key is the cue word, vals are the list of associated words
with jsonlines.open(zh_swow_test_data_fp, 'r') as reader:
    # collect all data into a list
    for obj in tqdm(reader):
        data_dict[obj['input']] = obj['output']

0it [00:00, ?it/s]

In [7]:
system_prompt_1 = "您是一款专为全面探索词语关联而设计的高级语言模型。"
system_prompt_2 = "你是人工智能助手"

instruction_prompt_template_1 = "给定一个提示词，你的任务是生成一个与该提示词相关联的全面词汇列表。目标是尽可能涵盖所有相关的语境、用法和含义，避免重复相似的概念。这些词共同提供对所有重要关联的广泛而深刻的表示。专注于揭示与提示词相关的常见和独特的方面，以确保对潜在关联进行平衡而彻底的探索。词语应彼此不同。你的回答只能是相关联的词语列表。不要生成受其他词语存在影响的词语，而是专注于提示词本身\n 提示词：{}"

instruction_prompt_template_2 = "我们来玩一个词联想实验,给你一个词,你告诉我你立马的联想词有哪些：{}. 请以列表形式返回。"



In [43]:
from concurrent.futures import ThreadPoolExecutor


In [47]:




def generate_comparing_table_for_cue_word(cue_word):
    # step 1: find the GT associated words
    gt_associated_words = data_dict[cue_word]
    
    pbar = tqdm(total=2, desc=f"Processing cue word: {cue_word}")
    # step 2: prompt type 1 
    messages = [
        {"role": "system", "content": system_prompt_1},
        {"role": "user", "content": instruction_prompt_template_1.format(cue_word)}
    ]
    
    tasks = [('ft',), ('raw',), ('power',)]
    
    def run_task(args):
        return get_reasoning_and_content(messages, *args)
    
    # 并行执行任务
    with ThreadPoolExecutor(max_workers=3) as executor:
        # 提交所有任务
        futures = [executor.submit(run_task, arg) for arg in tasks]
        
        # 按顺序获取结果（与任务列表顺序一致）
        results = [future.result() for future in futures]
        

    # 解包结果
    (type_1_reasoning_ft, type_1_content_ft), \
    (type_1_reasoning_raw, type_1_content_raw), \
    (type_1_reasoning_power, type_1_content_power) = results
    
    pbar.update(1)
    
    # step 3: prompt type 2
    messages = [
        {"role": "system", "content": system_prompt_2},
        {"role": "user", "content": instruction_prompt_template_2.format(cue_word)}
    ]
    
    # 并行执行任务
    tasks = [('ft',), ('raw',), ('power',)]
    def run_task(args):
        return get_reasoning_and_content(messages, *args)
    with ThreadPoolExecutor(max_workers=3) as executor:
        # 提交所有任务
        futures = [executor.submit(run_task, arg) for arg in tasks]
        
        # 按顺序获取结果（与任务列表顺序一致）
        results = [future.result() for future in futures]
    # 解包结果
    (type_2_reasoning_ft, type_2_content_ft), \
    (type_2_reasoning_raw, type_2_content_raw), \
    (type_2_reasoning_power, type_2_content_power) = results
    
    pbar.update(1)
    pbar.close()
    
    
    # step 4: print the comparing table
    html_table = f"""
<table border="1" style="border-collapse: collapse; width: 100%;">
    <tr>
        <th>Cue Word</th>
        <th>Ground Truth Associated Words</th>
        <th>Prompt Type</th>
        <th>Model Type</th>
        <th>Generated Content</th>
    </tr>
    <tr>
        <td rowspan="6">{cue_word}</td>
        <td rowspan="6">{gt_associated_words}</td>
        <td rowspan="3">Complex Type</td>
        <td>Qwen2.5 + SWOW-ZH</td>
        <td>{type_1_content_ft}</td>
    </tr>
    <tr>
        <td>Vanilla Qwen2.5</td>
        <td>{type_1_content_raw}</td>
    </tr>
    <tr>
        <td>Qwen3 32B (Powerful)</td>
        <td>{type_1_content_power}</td>
    </tr>
    <tr>
        <td rowspan="3">Simple Type</td>
        <td>Qwen2.5 + SWOW-ZH<</td>
        <td>{type_2_content_ft}</td>
    </tr>
    <tr>
        <td>Vanilla Qwen2.5</td>
        <td>{type_2_content_raw}</td>
    </tr>
    <tr>
        <td>Qwen3 32B (Powerful)</td>
        <td>{type_2_content_power}</td>
    </tr>
</table>"""
    return HTML(html_table)
    

In [49]:
gct= generate_comparing_table_for_cue_word

In [ ]:
gct("守望先锋")

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
gct("仍然")

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [36]:

gct('烤鸭')

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [37]:
gct('负责')

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [38]:
gct('很高')

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [39]:
gct('残缺')

Processing messages:   0%|          | 0/6 [00:00<?, ?it/s]

In [46]:
gct('优越')

In [48]:
gct('世事')

In [50]:
gct('扶持')

Processing cue word: 扶持:   0%|          | 0/2 [00:00<?, ?it/s]

In [51]:
gct('直男癌')

Processing cue word: 直男癌:   0%|          | 0/2 [00:00<?, ?it/s]

In [52]:
gct('急')

Processing cue word: 急:   0%|          | 0/2 [00:00<?, ?it/s]

In [53]:
gct('广场舞')

Processing cue word: 广场舞:   0%|          | 0/2 [00:00<?, ?it/s]

In [54]:
gct('罐头')

Processing cue word: 罐头:   0%|          | 0/2 [00:00<?, ?it/s]

In [55]:
gct('长时间')

Processing cue word: 长时间:   0%|          | 0/2 [00:00<?, ?it/s]

In [56]:
gct('不佳')

Processing cue word: 不佳:   0%|          | 0/2 [00:00<?, ?it/s]

In [59]:
gct('白色')

Processing cue word: 白色:   0%|          | 0/2 [00:00<?, ?it/s]

In [60]:
gct('负')

Processing cue word: 负:   0%|          | 0/2 [00:00<?, ?it/s]

In [61]:
gct('新人')

Processing cue word: 新人:   0%|          | 0/2 [00:00<?, ?it/s]

In [62]:
gct('岗位')

Processing cue word: 岗位:   0%|          | 0/2 [00:00<?, ?it/s]

In [64]:
gct('汪')

Processing cue word: 汪:   0%|          | 0/2 [00:00<?, ?it/s]

In [65]:
gct('一个人')

Processing cue word: 一个人:   0%|          | 0/2 [00:00<?, ?it/s]

In [66]:
gct('民国')

Processing cue word: 民国:   0%|          | 0/2 [00:00<?, ?it/s]

In [67]:
gct('火车站')

Processing cue word: 火车站:   0%|          | 0/2 [00:00<?, ?it/s]

In [68]:
gct('印度')

Processing cue word: 印度:   0%|          | 0/2 [00:00<?, ?it/s]

In [69]:
gct('劳作')

Processing cue word: 劳作:   0%|          | 0/2 [00:00<?, ?it/s]

In [70]:
gct('胖子')

Processing cue word: 胖子:   0%|          | 0/2 [00:00<?, ?it/s]

In [71]:
gct('孔')

Processing cue word: 孔:   0%|          | 0/2 [00:00<?, ?it/s]

In [72]:
gct('租房')

Processing cue word: 租房:   0%|          | 0/2 [00:00<?, ?it/s]

In [73]:
gct('成人')

Processing cue word: 成人:   0%|          | 0/2 [00:00<?, ?it/s]

In [74]:
gct('冰水')

Processing cue word: 冰水:   0%|          | 0/2 [00:00<?, ?it/s]

In [75]:
gct('辣椒')

Processing cue word: 辣椒:   0%|          | 0/2 [00:00<?, ?it/s]

In [76]:
gct('宅')

Processing cue word: 宅:   0%|          | 0/2 [00:00<?, ?it/s]

In [77]:
gct('嗨')

Processing cue word: 嗨:   0%|          | 0/2 [00:00<?, ?it/s]

In [78]:
gct('优惠券')

Processing cue word: 优惠券:   0%|          | 0/2 [00:00<?, ?it/s]